<a href="https://colab.research.google.com/github/singhprinceraj2210/models/blob/main/hybrid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gensim

import numpy as np
import pandas as pd
import re
import nltk
import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from gensim.models import Word2Vec

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
     Input,
    Conv1D,
    MaxPooling1D,
    LSTM,
    Dense,
    Dropout,
    BatchNormalization
)

from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 30.6 MB/s eta 0:00:00


In [ ]:
df = pd.read_csv("/content/IMDB Dataset.csv")  # columns: text, label

print(df.head())

                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive


In [ ]:
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

# Keep important negation words
negation_words = {'not', 'no', 'nor', 'never'}

stop_words = stop_words - negation_words


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
def clean_text(text):

    text = str(text).lower()

    # remove html tags
    text = re.sub(r'<.*?>', '', text)

    # remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # tokenization
    words = text.split()

    # remove stopwords
    words = [w for w in words if w not in stop_words]

    return words


# Apply preprocessing
df['tokens'] = df['review'].apply(clean_text)


In [ ]:
from gensim.models import Word2Vec

loaded_model = Word2Vec.load(
    "/content/drive/MyDrive/word2vec_model.model"
)

print("Model loaded successfully!")

Model loaded successfully!


In [ ]:
def vectorize_text(tokens, model):

    vectors = []

    for word in tokens:

        if word in model.wv:
            vectors.append(model.wv[word])

    # Handle empty sentences
    if len(vectors) == 0:
        vectors.append(np.zeros(model.vector_size))

    return vectors


X = []

for tokens in df['tokens']:

    vec = vectorize_text(tokens, loaded_model)

    X.append(vec)

In [ ]:
MAX_LEN = 100

X = pad_sequences(
    X,
    maxlen=MAX_LEN,
    dtype='float32',
    padding='post',
    truncating='post'
)

print("\nX Shape:")
print(X.shape)


X Shape:
(50000, 100, 100)


In [ ]:
y = df['sentiment'].map({
    'positive': 1,
    'negative': 0
}).values

print("\nY Shape:")
print(y.shape)


Y Shape:
(50000,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nTrain Shape:", X_train.shape)
print("Test Shape:", X_test.shape)



Train Shape: (40000, 100, 100)
Test Shape: (10000, 100, 100)


In [ ]:
model_hybrid = Sequential()

EMBEDDING_DIM = loaded_model.vector_size

# Input Layer
model_hybrid.add(
    Input(shape=(MAX_LEN, EMBEDDING_DIM))
)

In [ ]:
model_hybrid.add(
    Conv1D(
        filters=128,
        kernel_size=5,
        activation='relu'
    )
)

# Batch Normalization
model_hybrid.add(BatchNormalization())

# Pooling Layer
model_hybrid.add(
    MaxPooling1D(pool_size=2)
)

In [ ]:
model_hybrid.add(
    LSTM(
        128,
        return_sequences=False
    )
)

# Dropout
model_hybrid.add(Dropout(0.5))

# Dense Layer
model_hybrid.add(Dense(64, activation='relu'))

# Dropout
model_hybrid.add(Dropout(0.5))

# Output Layer
model_hybrid.add(Dense(1, activation='sigmoid'))



In [ ]:
model_hybrid.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)


In [ ]:
model_hybrid.summary()


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 96, 128)        │        64,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 96, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 48, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 204,545 (799.00 KB)

 Trainable params: 204,289 (798.00 KB)

 Non-trainable params: 256 (1.00 KB)

In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

In [ ]:
history = model_hybrid.fit(
    X_train,
    y_train,
    epochs=10,
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)


Epoch 1/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 117s 187ms/step - accuracy: 0.9086 - loss: 0.2340 - val_accuracy: 0.8652 - val_loss: 0.3669
Epoch 2/10
625/625 ━━━━━━━━━━━━━━━━━━━━ 116s 185ms/step - accuracy: 0.9231 - loss: 0.1988 - val_accuracy: 0.8525 - val_loss: 0.3889


In [ ]:
loss, accuracy = model_hybrid.evaluate(
    X_test,
    y_test
)

print("\n================================")
print("TEST ACCURACY")
print("================================")

print("Accuracy:", accuracy)


313/313 ━━━━━━━━━━━━━━━━━━━━ 13s 42ms/step - accuracy: 0.8652 - loss: 0.3669

TEST ACCURACY
Accuracy: 0.8651999831199646


In [ ]:
y_pred_prob = model_hybrid.predict(X_test)

y_pred = (
    y_pred_prob > 0.5
).astype(int).reshape(-1)

313/313 ━━━━━━━━━━━━━━━━━━━━ 15s 45ms/step


In [ ]:
print("\n================================")
print("CLASSIFICATION REPORT")
print("================================")

print(
    classification_report(
        y_test,
        y_pred
    )
)


CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.86      0.87      0.87      5000
           1       0.87      0.86      0.86      5000

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000



In [ ]:
print("\n================================")
print("CONFUSION MATRIX")
print("================================")

print(
    confusion_matrix(
        y_test,
        y_pred
    )
)


CONFUSION MATRIX
[[4351  649]
 [ 699 4301]]


In [ ]:
model_hybrid.save(
    '/content/drive/MyDrive/cnn_lstm_hybrid_model.keras'
)


In [ ]:
def predict_sentiment(text):

    # clean text
    tokens = clean_text(text)

    # vectorize
    vec = []

    for word in tokens:

        if word in loaded_model.wv:
            vec.append(loaded_model.wv[word])

    # Handle empty sentence
    if len(vec) == 0:
        vec.append(np.zeros(EMBEDDING_DIM))

    # padding
    vec = pad_sequences(
        [vec],
        maxlen=MAX_LEN,
        dtype='float32',
        padding='post',
        truncating='post'
    )
    # prediction
    pred = model_hybrid.predict(vec)[0][0]

    if pred > 0.5:
        return f"Positive Sentiment ({pred:.4f})"

    else:
        return f"Negative Sentiment ({pred:.4f})"



In [ ]:
print(
    predict_sentiment(
        "this movie was absolutely amazing"
    )
)

print(
    predict_sentiment(
        "this movie is not good"
    )
)

print(
    predict_sentiment(
        "worst movie ever"
    )
)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
Positive Sentiment (0.9419)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
Positive Sentiment (0.7224)
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 66ms/step
Negative Sentiment (0.0223)
